# Train SomaticXGB

End-to-end notebook for reproducing the SomaticXGB model (run from `3.SomaticXGB-training/`):

1. **Build a labeled feature matrix** from HG001-003 PosMix runs that already finished **force-calling** (`"features"` in pipeline 2).
2. **5-fold CV grid search** over `max_depth`, `eta`, `nrounds` (PR-AUC).
3. **Train the final model** with the selected (or manuscript) hyperparameters and save it.

Feature extraction uses `SomaticXGB_features.R` / `build_feature_matrix.R` in this same directory. Outputs (`feature_matrix.tsv.gz`, CV tables, importance, retrained model) are written here as well.

## Prerequisite

Pipeline-2 `data[].id` is a **platform name** (e.g. `SURFSeq_5000`). Run **each training pair in its own directory** with that pair’s pipeline-2 `config.json`. Step 3 reads those step-2 configs (no separate training config).

Each pair directory must contain force-calling outputs for every `id` × `tumor_reads` in its config:

- `mutect_a_vraw/{id}_reads{reads}M.raw.vcf.gz`
- `varscan/{id}_reads{reads}M.raw.snp.sort.vcf.gz`
- `varscan_novo/{id}_reads{reads}M.raw.snp.vcf.rc.tsv.gz`


## 0. Paths and packages

Set `pair_dirs` to the per-pair working directories you created when you ran pipeline 2. The six paths below only show the manuscript pair IDs (`12`, `13`, `21`, `23`, `31`, `32`). Replace them with your paths. This notebook does not create those directories and does not run alignment, PosMix, or calling.

All other outputs stay in this directory (`3.SomaticXGB-training/`).


In [ ]:
library(data.table)
library(xgboost)
library(caret)
library(PRROC)

# Working directory should be 3.SomaticXGB-training/ (where this notebook lives).
stopifnot(file.exists("build_feature_matrix.R"), file.exists("SomaticXGB_features.R"))

# Replace each entry with a pipeline-2 working directory you created
# (config.json + force-calling outputs). These examples are not real paths.
pair_dirs <- c(
  "/path/to/your/train/12",
  "/path/to/your/train/13",
  "/path/to/your/train/21",
  "/path/to/your/train/23",
  "/path/to/your/train/31",
  "/path/to/your/train/32"
)

feature_matrix <- "feature_matrix.tsv.gz"
cv_results_file <- "cv_results.tsv"
med_auc_file <- "med_auc.tsv"
importance_out <- "importance.tsv"
model_out <- "SomaticXGB_retrained"

nthread <- 8
set.seed(820)

features <- c(
  "DP", "ECNT", "MBQ_REF", "MBQ_ALT", "MFRL_REF", "MFRL_ALT", "MMQ_REF", "MMQ_ALT",
  "MPOS", "NALOD", "NLOD", "TLOD",
  "avg_mapping_quality", "avg_basequality", "avg_pos_as_fraction",
  "avg_num_mismatches_as_fraction", "avg_sum_mismatch_qualities"
)

# Manuscript hyperparameters (also the top median PR-AUC in med_auc_0820.xls)
manuscript_params <- list(max_depth = 7, eta = 0.1, nrounds = 400)

getwd()


## 1. Build the feature matrix from force-calling outputs

Calls `build_feature_matrix.R` on the pair directories above. Each directory’s pipeline-2 `config.json` supplies `id`, `tumor_reads`, and `posfreq`.

The feature matrix is not in the repository. This cell builds it from the force-calling outputs of pipeline 2. A `feature_matrix.tsv.gz` already written by this builder can be read with `fread` instead of rebuilding.


In [ ]:
if (!length(pair_dirs) || any(startsWith(pair_dirs, "/path/to/your/"))) {
  stop("Edit pair_dirs to the pipeline-2 working directories you created. This notebook does not fill in those paths.")
}
missing_dirs <- pair_dirs[!dir.exists(pair_dirs)]
if (length(missing_dirs)) {
  stop("pair_dirs not found:\n  ", paste(missing_dirs, collapse = "\n  "))
}

cmd <- paste(
  "Rscript",
  shQuote("build_feature_matrix.R"),
  paste(shQuote(pair_dirs), collapse = " "),
  shQuote(feature_matrix)
)
message("Running: ", cmd)
status <- system(cmd)
if (status != 0) stop("build_feature_matrix.R failed with status ", status)

dtm <- fread(feature_matrix)
message("Rows: ", nrow(dtm), "  positives: ", sum(dtm$label == 1), "  negatives: ", sum(dtm$label == 0))
if (!is.null(dtm$group)) print(table(dtm$group, useNA = "ifany"))
head(dtm)


## 2. Hyperparameter search with 5-fold CV

Grid matches the original tuning notebook. Metric: precision–recall AUC on each held-out fold; summarize by **median** PR-AUC per parameter set.


In [ ]:
max_depths <- c(3, 5, 7, 9)
etas <- c(0.01, 0.05, 0.1, 0.2)
nrounds_list <- c(100, 200, 400)
k_folds <- 5

folds <- createFolds(dtm$label, k = k_folds, list = TRUE, returnTrain = FALSE)
results <- data.table(max_depth = integer(), eta = numeric(), nrounds = integer(),
                      k = integer(), pr_auc = numeric())

for (max_depth in max_depths) {
  for (eta in etas) {
    for (nrounds in nrounds_list) {
      for (i in seq_len(k_folds)) {
        test_idx <- folds[[i]]
        dt_train <- dtm[-test_idx]
        dt_test <- dtm[test_idx]
        dtrain <- xgb.DMatrix(
          data = data.matrix(dt_train[, features, with = FALSE]),
          label = dt_train$label
        )
        model <- xgboost(
          data = dtrain,
          max.depth = max_depth,
          eta = eta,
          nthread = nthread,
          nrounds = nrounds,
          objective = "binary:logistic",
          verbose = 0
        )
        pred <- predict(model, data.matrix(dt_test[, features, with = FALSE]))
        pr <- pr.curve(scores.class0 = pred, weights.class0 = dt_test$label, curve = FALSE)
        row <- data.table(max_depth = max_depth, eta = eta, nrounds = nrounds,
                          k = i, pr_auc = pr$auc.integral)
        print(row)
        results <- rbind(results, row)
      }
    }
  }
}

fwrite(results, cv_results_file, sep = "\t")
med_auc <- results[, .(med = median(pr_auc)), by = .(max_depth, eta, nrounds)]
setorder(med_auc, -med)
fwrite(med_auc, med_auc_file, sep = "\t")
med_auc


## 3. Choose hyperparameters and train the final model

By default we use the **manuscript** settings (`max.depth=7`, `eta=0.1`, `nrounds=400`), which also ranked at the top of the original median-AUC table. Set `use_cv_best <- TRUE` to take the best row from the grid above instead.

The final model is fit on **all** training rows.


In [ ]:
use_cv_best <- FALSE

if (use_cv_best) {
  best <- med_auc[1]
  params <- list(max_depth = best$max_depth, eta = best$eta, nrounds = best$nrounds)
} else {
  params <- manuscript_params
}
print(params)

dtrain <- xgb.DMatrix(
  data = data.matrix(dtm[, features, with = FALSE]),
  label = dtm$label
)
xgb_model <- xgboost(
  data = dtrain,
  max.depth = params$max_depth,
  eta = params$eta,
  nthread = nthread,
  nrounds = params$nrounds,
  objective = "binary:logistic",
  verbose = 1
)

xgb.save(xgb_model, fname = model_out)
message("Saved model: ", model_out)

imp <- xgb.importance(feature_names = features, model = xgb_model)
fwrite(imp, importance_out, sep = "\t")
imp


## 4. Quick sanity check (in-sample PR curve)

For an honest estimate rely on the CV results above. Holding out HG004-005 is done with `4.SomaticXGB-predict/` after running pipeline 2 on the test pairs with `"features"`.


In [ ]:
pred <- predict(xgb_model, data.matrix(dtm[, features, with = FALSE]))
pr <- pr.curve(scores.class0 = pred, weights.class0 = dtm$label, curve = TRUE)
print(pr$auc.integral)
plot(pr, main = "In-sample Precision-Recall (training matrix)",
     xlab = "Recall", ylab = "Precision")

## Next step

Point `4.SomaticXGB-predict` at the new model in this directory:

```json
"model": "../3.SomaticXGB-training/SomaticXGB_retrained"
```

Or replace / compare against the released `../models/SomaticXGB`.
